NOTES CELL (TO BE DELETED PRIOR TO SUBMISSION)

- Mohammed: Doing interpolation. Taking in an equilibria file. Outputting interpolated magnetic field. Needs to figure out what to use as input file (solved equilibria or not) and do some reading to figure out how much of interpolation should be local (written by ourselves) and how much should be from DESC. For example, an MHD solver should be from DESC, if we decide to use that.
- John: Doing integration. Is responsible for taking the interpolated magnetic field and then finding the magnetic field lines, and then the magnetic field bounce points, and then the bounce integration of an arbitrary quantity.

Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as any collaborators you worked with:

In [2]:
COLLABORATORS = "Mohammed Haque and John Labbate"

## To receive credit for this assignment, you must also fill out the [AI Use survey](https://forms.gle/ZhR5k8TdAeN8rj4CA)


---

In [3]:
%matplotlib inline
%precision 16
import numpy
import matplotlib.pyplot as plt
import pandas as pd

# Final Project

This notebook will provide a brief structure and rubric for presenting your final project. 

The purpose of the project is 2-fold
* To give you an opportunity to work on a problem you are truly interested in (as this is the best way to actually learn something)
* To demonstrate to me that you understand the overall workflow of problem solving from problem selection to implementation to discussion 

You can choose any subject area that interests you as long as there is a computational component to it.  However, please do not reuse projects or homeworks you have done in other classes.  This should be **your** original work.

**You can work in teams, but clearly identify each persons contribution** and every team member should hand in their own copy of the notebook.

### Structure
There are 5 parts for a total of 100 points that provide the overall structure of a mini research project.

* Abstract
* Introduction and Problem Description
* Brief discussion of Computational approach and import of any additional packages
* Implementation including tests
* Discussion of results and future directions

For grading purposes, please try to make this notebook entirely self contained. 

The project is worth about 2 problem sets and should be of comparable length (please: I will have about 100 of these to read and I am not expecting full 10 page papers).  The actual project does not necessarily have to work but in that case you should demonstrate that you understand why it did not work and what steps you would take next to fix it.

Have fun

## Abstract [10 pts]

Provide a 1-2 paragraph abstract of the project in the style of a research paper.  The abstract should contain

* A brief description of the problem
* A brief justification describing why this problem is important/interesting to you
* A general description of the computational approach
* A brief summary of what you did and what you learned


YOUR ANSWER HERE

## Introduction [15 pts]

In ~4-5 paragraphs, describe 
* The general problem you want to solve
* Why it is important and what you hope to achieve.

Please provide basic **references**, particularly if you are reproducing results from a paper. Also include any basic equations you plan to solve. 

Please use proper spelling and grammar. 

YOUR ANSWER HERE

### References

YOUR ANSWER HERE

## Computational  Methods [10 pts]

Describe the specific approach you will take to solve some concrete aspect of the general problem. 

You should  include all the numerical or computational methods you intend to use.  These can include methods or packages  we did not discuss in class but provide some reference to the method. You do not need to explain in detail how the methods work, but you should describe their basic functionality and justify your choices. 




YOUR ANSWER HERE

**If you need to install or import any additional python packages,  please provide complete installation instructions in the code block below**


In [9]:
# Provide complete installation or import information for external packages or modules here e.g.

# For the codes utilized for this project, all installations are done in the regime of a conda environment, which requires the installation of Anaconda or Miniconda.
# Utilize the instructions at https://www.anaconda.com/docs/getting-started/anaconda/install for installing Anaconda or Miniconda.

'''
##### Instructions to install the DESC code #####

# Run the following commands in a new terminal on MacOS or a Linux computing cluster:
git clone https://github.com/PlasmaControl/DESC.git
cd DESC
conda create --name APMA4300 'python>=3.10, <=3.13'
conda activate APMA4300
conda install pip
pip install --editable .
'''
import desc.io

'''
##### Instructions to install JAX #####

# Run the following commands in a new terminal
conda activate APMA4300
pip install jax==0.7.2
'''
import jax
import jax.numpy as jnp

## Implementation [50 pts]

Use the Markdown and Code blocks below to implement and document your methods including figures.  Only the first markdown block will be a grading cell but please add (not copy) cells in this section to organize your work. 

Please make the description of your problem readable by interlacing clear explanatory text with code (again with proper grammar and spelling). 
All code should be well described and commented.

For at least one routine you code below, you should provide a test block (e.g. using `numpy.testing` routines, or a convergence plot) to validate your code.  

An **important** component of any computational paper is to demonstrate to yourself and others that your code is producing correct results.

We will begin with a DESC stellarator equilibrium file. This is a file that holds information of a magnetohydrodynamic (MHD) balanced stellarator configuration from which DESC has methods to calculate specific quantities [1]. A good starting point for this project is the magnetic field magnitude $B$ at each point on a specified grid within a stellarator, which can be calculated from this equilibrium file. We will use a common 3D coordinate system to describe a stellarator with closed flux surfaces $\psi,\alpha,\zeta$, which is a magnetic coordinate system where $\psi$ is the toroidal flux label, $\alpha$ is the label of a magnetic field line, and $\zeta$ is the toroidal angle [2]. In DESC, instead of $\psi$, $\rho$ is used. These two coordinates are connected:

$$ \rho = \sqrt{\frac{\psi}{\psi_a}} $$

where $\psi_a$ is the toroidal flux of the last closed flux surface. Note we are assuming closed toroidal flux surfaces exist in our stellarator equilibrium. The equilibrium file we will use is one of the provided example equilibria from DESC's tutorials. If the install above is followed exactly, the code below will access the same equilibrium with no extra work required except the specification of the path to the "DESC" repository top directory.

We use this magnetic coordinates system because we assume energetic particles have low enough energies such that their orbits generally follow magnetic field lines. If the drifting motion (non-gyromotion) was large, these particles would not follow magnetic field lines.

JAX is utilized for the implementation of this code in order to maintain compatibility with gradient methods in JAX. This is because these steps would likely be used in an optimization routine.

In [ ]:
# Start with a DESC equilibrium file
eq = desc.io.load('qs_initial_guess.h5')

In [18]:
# Specify a psi, alpha, zeta grid to work with
def make_grid(
        eq,
        rho = jnp.linspace(0,1,10),
        alpha = jnp.linspace(0,2*jnp.pi,10),
        zeta = jnp.linspace(0,2*jnp.pi,100)
        ):
    return eq._get_rtz_grid(
        radial = rho,
        poloidal = alpha,
        toroidal = zeta,
        coordinates = 'raz',
    )

In [ ]:
# Calculate the magnetic field magnitude B on the specified grid above
def compute_from_eq(eq,grid):
    return eq.compute("B",grid=grid), eq.compute("B_sup_zeta",grid=grid)

With the magnetic field known as discrete points in space, we can treat $B$ as a one-dimensional function for each $(\psi,\alpha)$ combination. We will want to perform an integration along a magnetic field line to get what is known as a "bounce integrated" quantity, which will be described in more detail shortly. This integration will be done in two methods. First, we will integrate analytically a fit polynomial function to the 1D $B$. Second, we will integrate numerically using the grid points specified along each magnetic field line. Third, our results from these two methods will be compared with the master branch bounce integration capabilities of DESC.

Below, we complete the interpolation of the magnetic field for the analytic bounce integration. For an optimization loop - where this code would commonly be used - many optimizers require the second derivative of relavent quantities to be well behaved. In this class, we learned a few interpolation methods. This project will apply cubic splines and chebyshev polynomials interpolate $B$.

In [ ]:
def polyfit_diff(x, y, deg): # jax.polyfit but differentiable 
    """
    Differentiable polynomial fit via least-squares.

    Args:
        x: shape (n,)
        y: shape (n,)
        deg: int, polynomial degree

    Returns:
        coeffs: shape (deg+1,), highest-power first
    """
    # Construct Vandermonde matrix (n, deg+1)
    A = jnp.vander(x, N=deg+1, increasing=False)

    # Solve least squares: minimize ||A c - y||^2
    coeffs, residuals, rank, s = jnp.linalg.lstsq(A, y, rcond=None)
    return coeffs

In [ ]:
# Interpolation to form integrand
def affine_transform(x,a,b):
    """
    Map x from [a, b] to [-1, 1]

    Args:
        x: array
        a: lower bound of original interval
        b: upper bound of original interval
    """
    return 2 * (x - a) / (b - a) - 1

def interpolate(f_eval,x,method='cubic splines'):
    """
    Interpolate a function

    Args:
        f_eval: array of f evaluated at points in x interval
        x: x points at which f was evaluated at to form f_eval

    Returns:
        d_arr_full: array of polynomial's second derivatives at each point
    """
        
    if method == 'cubic splines':
        x_affine = affine_transform(x,0,1)
        N = len(x_affine) # degree of polynomial is the number of data points we have - 1
        A = jnp.zeros((N-2,N-2))
        b = jnp.zeros(N-2)
        for i in range(0,N-2):
            k = i+1
            A = A.at[i,i].set(2)
            if i+1<N-2:
                A = A.at[i,i+1].set(8)
            if i+2<N-2:
                A = A.at[i,i+2].set(2)
            b = b.at[i].set(6*(f_eval[k+1] - f_eval[k-1]))

        # Solve for the second derivative coefficients
        d_arr = jnp.linalg.solve(A,b)
        # use "natural splines", specify second derivative at each boundary to be zero
        d_arr_full = jnp.zeros((N,N))
        d_arr_full = d_arr_full.at[0].set(0) # Boundary derivative
        d_star = d_star.at[1:n-1].set(d_interior)  # Interior derivatives
        d_arr_full = d_arr_full.at[-1].set(0) # Boundary derivative

        def evaluate_interpolated(x_new):
            """
            Evaluate a function at a new point using cubic splines

            Args:
                x_new: array of new points to evaluate the function at
                x: x points at which f was evaluated at to form f_eval
                f_eval: array of f evaluated at points in x interval

            Returns:
                P: array of values of the function at x_new
            """

            # Compute cubic spline coefficients
            d_star = interpolate(f_eval,x,method='cubic splines')

            # Find which interval contains x_new
            # Use searchsorted to find the right interval
            idx = arr_max_diff(-1*abs(x - x_new))['max_i']
            
            results = jnp.zeros(x_new)
            for i in range(len(x_new)):
                k = idx[i]
                
                # Get interval data
                x_k = x_knots[k]
                x_kp1 = x_knots[k + 1]
                y_k = f_values[k]
                y_kp1 = f_values[k + 1]
                d_k = d_values[k]
                d_kp1 = d_values[k + 1]

                # # Get interval endpoints
                # x_k = x[idx]
                # x_kp1 = x[idx + 1]
                # y_k = f_eval[idx]
                # y_kp1 = f_eval[idx + 1]
                # d_k = d_star[idx]
                # d_kp1 = d_star[idx + 1]

                s = affine_transform(x,0,1)
                
                # Evaluate cubic polynomial using the formula from lecture 06
                P = ((1-s)**2 * (1 + 2*s) * y_k + s**2 * (3 - 2*s) * y_kp1 + s * (1-s)**2 * d_k - s**2 * (1-s) * d_kp1)
                results = results.at[i].set(P)
            return results
        return evaluate_interpolated

    else:
        raise ValueError("Invalid interpolation method") 

def arr_max_diff(arr):
    """
    Find the maximum of a 1D array (jax differentiable)

    Args:
        arr: 1D array

    Returns:
        dict with the index of max occurance and the value of the maximum
    """

    max_num = 99999999999999.9
    max_i = len(arr)+10 # this will throw an error if this function is not written correctly
    i_dumby = 0
    arr_dict = {'arr':arr,'i':i_dumby,'max_num':max_num,'max_i':max_i}
    def body_fun(i,arr_dict):
        cond_dict = {'arr':arr_dict['arr'],
                        'i':i,
                        'max_num':max_num,
                        'max_i':max_i}
        def tb_max(cond_dict):
            cond_dict['max_num'] = cond_dict['arr'][cond_dict['i']]
            cond_dict['max_i'] = cond_dict['i']
            return cond_dict
        def fb_max(cond_dict):
            return cond_dict
        return jax.lax.cond(cond_dict['arr'][i]<cond_dict['max_num'],tb_max,fb_max,cond_dict)
    out = jax.lax.fori_loop(0,len(arr),body_fun,arr_dict)
    return {'max_i':out['max_i'],'max_num':out['max_num']} # returns a dict with the index of max occurance and the value of the maximum

Since particles are assumed to be stuck to the magnetic field lines, the shape of the magnitude of the magnetic field curve along a field line will determine any mirroring or trapping that occurs for a particle. The magnitude of the magnetic field $B_c$ that causes a particle to turn around (an effect called mirroring [3]) is set by a parameter known as the pitch parameter $\lambda$.
$$
B_c = \frac{1}{\lambda}
$$
A routine is written in the following code cell to specify a 1D grid of $B_c$ depending on the maximum and minimum values of the magnetic field magnitude for one flux surface ($\rho$), magnetic field line ($\alpha$), and magnetic well combination. This routine ensures that each particle is "trapped" in that it mirrors back and forth within a magnetic well (or "barely-trapped" if there is only one mirror point).

In [ ]:
# Specify 1D grid for B_c
from desc.equilibrium.coords import get_rtz_grid

def get_Bc_grid(eq,rho,num_Bc):
    """
    Get B_crit values for each rho considered

    Args:
        eq: DESC equilibrium
        rho: array of rho values
        num_Bc: integer number of B_crit values to consider

    Returns:
        2D array of B_crit values to consider, shape: (rho,Bc)
    """
    rho_bc = rho
    alpha_bc = jnp.array([0])
    zeta_bc = jnp.linspace(0, 12 * jnp.pi, 50)
    grid_bc = get_rtz_grid(eq, rho_bc, alpha_bc, zeta_bc, coordinates="raz")
    data = eq.compute(["min_tz |B|", "max_tz |B|"], grid=grid_bc) # min,max |B| on each rho
    return jnp.linspace(
        jnp.mean(data['min_tz |B|']),
        jnp.mean(data['max_tz |B|']),
        num_Bc
    )

The next cell determines the spatial point at which particle mirroring occurs is recorded for each combination of $\rho,\alpha,B_c,$ and magnetic well along the relevant field line. For simplicity, we will use the DESC method to do this. A method could be written using a root-finder, however, it will likely make this project too long.

In [ ]:
# Identifying bounce/mirror points
from desc.integrals.bounce_integral import Bounce1D
from desc.integrals.quad_utils import (
    automorphism_sin,
    get_quadrature,
    grad_automorphism_sin,
)
from orthax.legendre import leggauss
from desc.grid import LinearGrid

def get_bounce_points(eq,grid,Bcs,num_quad=32):
    """
    Find zeta bounce points for each rho,alpha,well,Bc considered

    Args:
        eq: DESC equilibrium
        grid: relevant grid
        Bcs: array of B_crits to consider
        num_quad: integer number of quadrature points to consider

    Returns:
        array of zeta bounce point values to consider for each rho,alpha,well,Bc. shape (rho,alpha,well,Bc,zeta one and two)
    """

    # DESC method
    rho = grid.nodes[:,0]
    _keys_1dr = ["iota", "iota_r", "min_tz |B|", "max_tz |B|"]
    _grid_1dr = LinearGrid(
            rho=rho, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP, sym=eq.sym
    )
    data = {
        key: grid.copy_data_from_other(data[key], _grid_1dr)
        for key in _keys_1dr
    }
    quad = (
        get_quadrature(
            leggauss(num_quad),
            (automorphism_sin, grad_automorphism_sin),
        )
    )
    bounce = Bounce1D(grid, data, quad, is_reshaped=True)
    return bounce.points(Bcs, num_well=num_well)


Once each bounce point/mirror point has been identified, the actual integration over a period of mirroring may be calculated.
$$
\langle f \rangle_b = \frac{2}{v\tau_b}\int_{l_0}^{l_f}\frac{f}{\sqrt{1-\frac{B}{B_c}}}dl
$$
This integral will be calculated for each $\rho,\alpha,B_c,$ and magnetic well combination using the Gauss-Legendre quadrature method. The quadrature weights and associated quantities are defined in the following cell.

In [ ]:
# Create quadratures
def get_quad_leggaus(rho,Bcs,points,num_quad=32):
    z1 = points[0][:][:][:][:] # rho, alpha, Bc, well
    z2 = points[1][:][:][:][:] # rho, alpha, Bc, well
    x_ref, w_ref = leggauss(num_quad)
    zeta = 0.5*(z1 + z2) + 0.5*(z2 - z1) * x_ref # shape (n_quad, rho, alpha, Bc, well)
    dzetadx  = 0.5 * (z2 - z1) # shape (n_quad, rho, alpha, Bc, well)
    return zeta, w_ref, dzetadx  # integrate: sum(w_ref * f(zeta) * x_to_zeta_jac)

To now develop the integral numerically, we must keep all our variable changes in order. We can write this integral shown above with $g = \frac{f}{\sqrt{1-\frac{B}{B_c}}}$ in a way that allows the use of the quantities we have considered in this document so far:

$$
\int_{l_0}^{l_f}g(l)dl = \int_{\zeta_1}^{\zeta_2}g(\zeta)\frac{dl}{d\zeta}d\zeta = \int_{x_1}^{x_2}g(x)\frac{dl}{d\zeta}\frac{d\zeta}{dx}dx 
$$

where $x=[-1,1]$ defines the grid of quadratures.

$g$ is some callable function that will be defined later in this code. 

$\frac{d\zeta}{dx}$ is the Jacobian from $\zeta$ to $x$ which has been defined in the get_quad_leggaus() method above.

$\frac{dl}{d\zeta}$ is known from [2] to be:

$$
\frac{dl}{d\zeta} = \left| \frac{\mathbf{B}}{\mathbf{B} \cdot \mathbf{\nabla} \zeta} \right| = \frac{|\mathbf{B}|}{|\mathbf{B^\zeta}|} = \frac{1}{|\mathbf{b^\zeta}|}
$$

These parameters may be calculated at the quadrature points using the following code.

In [ ]:
def compute_Bs_quad(eq,grid,rho,points,zeta_quad):
    """

    Args:


    Returns:
    """
    B, B_sup_zeta = compute_from_eq(eq=eq,grid=grid)
    zeta_knots = grid.nodes[:, 2] # zeta coordinates

    # Find B at quadrature points
    B_spline = interpolate(B, zeta_knots, method='cubic splines') # Build B spline
    B_quad = B_spline(x_new=zeta_quad) # B at quadrature points

    # Find B_sup_zeta at quadrature points
    B_sup_zeta_spline = interpolate(B_sup_zeta, zeta_knots, method='cubic splines') # Build B_sup_zeta spline
    B_sup_zeta_quad = B_sup_zeta_spline(x_new=zeta_quad) # B_sup_zeta at quadrature points

    return B_quad, B_sup_zeta_quad

In [ ]:
# Integrate!
def bounce_integrate(eq,grid,f):
    """
    Integrate f over a bounce period

    Args:
        eq: DESC equilibrium
        grid: relevant user-specifiedgrid
        f: callable function to be integrated

    Returns:
        array of bounce integrated values of f size (rho,alpha,Bc,well)
    """

    rho = grid.nodes[:, 0]
    Bcs = get_Bc_grid(eq,rho,num_Bc=5) # determine critical magnetic field values
    bounce_points = get_bounce_points(eq,grid,Bcs) # determine bounce points based on B_crits
    zeta_quad, w_ref, dzetadx = get_quad_leggaus(rho,Bcs,bounce_points) # determine zeta quadrature points
    B_quad, B_sup_zeta_quad = compute_Bs_quad(eq,grid,rho,Bcs,bounce_points,zeta_quad) # determine B and B_sup_zeta at quadrature points
    dldzeta = B_quad/B_sup_zeta_quad
    return f(zeta_quad) * dldzeta * dzetadx * w_ref

$$\textbf{TESTING}$$

Just like in DESC, this code will provide an example integration quantity to consider. This quantity will be used to test this code.

In [ ]:
# Example function to be considered for bounce integration
from desc.utils import safediv
f_test = lambda data, B, pitch: safediv(2.0, jnp.sqrt(jnp.abs(1 - pitch * B))) # test function, this is particle velocity * time
grid = make_grid(eq=eq) # grid to use for both methods

In [ ]:
# Useful definitions
num_quad = 32 # number of quadrature points to use. Always using 32 for this project
num_Bc = 10 # number of critical magnetic field values to use. Always using 10 for this project
num_well = 3 # number of magnetic wells to consider. Always use 3 for this project

In [ ]:
# Using purely DESC methods to test bounce integration (not yet running)
def DESC_test():
    _keys_1dr = ["iota", "iota_r", "min_tz |B|", "max_tz |B|"]
    _grid_1dr = LinearGrid(
            rho=grid.nodes[:,0], M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP, sym=eq.sym
    )
    data = {
        key: grid.copy_data_from_other(data[key], _grid_1dr)
        for key in _keys_1dr
    } # doesn't really matter what is in data for the purposes of this project, but needs definition to fit into DESC code
    quad = (
        get_quadrature(
            leggauss(num_quad),
            (automorphism_sin, grad_automorphism_sin),
        )
    )
    Bcs_DESC = get_Bc_grid(eq,rho=grid.nodes[:,0],num_Bc=num_Bc)
    bounce = Bounce1D(grid, data, quad, is_reshaped=True)
    points = bounce.points(Bcs_DESC, num_well=num_well)
    return bounce.integrate(
        integrand=[f_test],
        pitch_inv=Bcs_DESC,
    )

In [ ]:
# Numpy tests: Matching the DESC default answer to our methods

# Using purely DESC methods to test bounce integration (now running)
f_bounceint_DESC = DESC_test()

# Using this project's methods to test bounce integration
f_bounceint_test = bounce_integrate(eq,grid,f_test) # nothing like a good ol' one liner!

# Numpy testing
print("Numpy testing on pre-determined grid")
numpy.testing.assert_allclose(actual=f_bounceint_test,desired=f_bounceint_DESC,rtol=1e-3)
print("Success!")

In [ ]:
# Extra test: Differentiable for optimization?
test = jax.grad(bounce_integrate(eq,grid,f_test))
print('Success!')

References for Implementation

[1] https://desc-docs.readthedocs.io/en/stable/variables.html \\\
[2] Reference describing magnetic coordinates, maybe Intro to Stellarators textbook \\\
[3] Reference deriving a magnetic mirror \\\

## Discussion [15 pts]

Evaluate the results of your project including 
* Why should I believe that your numerical results are correct (convergence, test cases etc)?
* Did the project work (in your opinion)?
* If yes:  what would be the next steps to try
* If no:  Explain why your approach did not work and what you might do to fix it.


YOUR ANSWER HERE